# Bronze Layer: Raw Data Ingestion

## Purpose
This notebook ingests raw solar sales and operations data into the Bronze layer of our Databricks Lakehouse. 

The Bronze layer enables traceability by preserving our untransformed source data.

## Source Tables
- `solar_leads`
- `solar_sales`
- `solar_installations`

## Bronze Tables
- `bronze_solar_leads`
- `bronze_solar_sales`
- `bronze_solar_installations`

## Design
The Bronze layer is the first stage of our medallion architecture. It includes the raw version of source data and transformations are minimal to preserve our data for processing downstream.


In [0]:
from datetime import datetime, timezone
from pyspark.sql import functions as F

# Source -> Bronze mapping (per the design above)
source_tables = {
    "solar_leads": "bronze_solar_leads",
    "solar_sales": "bronze_solar_sales",
    "solar_installations": "bronze_solar_installations",
}

# Single ingestion timestamp for this run (traceability)
run_ts = datetime.now(timezone.utc)

source_system_name = "solar_operations"

for source, bronze in source_tables.items():
    df = spark.table(source)

    # Bronze keeps source data as-is; only append traceability columns
    (df
        .withColumn("source_system", F.lit(source_system_name))
        .withColumn("ingested_at", F.lit(run_ts))
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(bronze))

    print(f"{source} -> {bronze}: {df.count()} rows ingested")

In [0]:
%sql
-- Validate: bronze row counts and columns must match the sources
WITH src AS (
    SELECT 'solar_leads' AS tbl, COUNT(*) AS rows FROM solar_leads
    UNION ALL
    SELECT 'solar_sales', COUNT(*) FROM solar_sales
    UNION ALL
    SELECT 'solar_installations', COUNT(*) FROM solar_installations
),
brz AS (
    SELECT 'solar_leads' AS tbl, COUNT(*) AS rows FROM bronze_solar_leads
    UNION ALL
    SELECT 'solar_sales', COUNT(*) FROM bronze_solar_sales
    UNION ALL
    SELECT 'solar_installations', COUNT(*) FROM bronze_solar_installations
)
SELECT
    s.tbl,
    s.rows AS source_rows,
    b.rows AS bronze_rows,
    (s.rows = b.rows) AS counts_match
FROM src s
JOIN brz b ON s.tbl = b.tbl
ORDER BY s.tbl